# Load UKHRD into this Lakehouse — from the CSV store

Same end result as `load_ukhrd.ipynb` — `ukhrd_codes`, `ukhrd_lists` and **one table per reference list** —
but the source is the **CSV files in the repository**, not the Parquet files on a release.

**Use this one when** you want bronze to be the publisher's own files, byte for byte, with the whole change
history behind them in git; when your rules say land raw text first; or when a release has not been cut yet.
**Use `load_ukhrd.ipynb` instead when** you want the smallest, fastest load — the Parquet files are typed and
about 40 KB together.

It downloads one zip of the repository, keeps the CSVs in `Files/bronze/ukhrd/<date>/` untouched, and builds
the tables from there.

**Where the data comes from.** Contains information from NHS England, licensed under the current version of
the Open Government Licence. Codes and descriptions are copied exactly as published in the NHS Data Model and
Dictionary (https://www.datadictionary.nhs.uk/). UKHRD is independent and not endorsed by NHS England; the
dictionary is the authority. Keep this credit with the data.

In [ ]:
import datetime
import shutil
import urllib.request
import zipfile

from pyspark.sql import functions as F

ZIP = "https://github.com/eamazon/ukhrd/archive/refs/heads/main.zip"
FILES = "/lakehouse/default/Files/"    # the Lakehouse this notebook is attached to
PREFIX = "ukhrd_"                      # table names: ukhrd_admission_method, ukhrd_codes, …
DAY = datetime.date.today().isoformat()
BRONZE = f"bronze/ukhrd/{DAY}/"        # the publisher's files, untouched

urllib.request.urlretrieve(ZIP, FILES + "ukhrd.zip")
with zipfile.ZipFile(FILES + "ukhrd.zip") as z:
    wanted = [n for n in z.namelist() if "/data/" in n and n.endswith(".csv")]
    for name in wanted:
        target = FILES + BRONZE + name.split("/data/", 1)[1]
        import os
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(name) as inside, open(target, "wb") as out:
            shutil.copyfileobj(inside, out)

print(len(wanted), "CSV files landed in Files/" + BRONZE)

In [ ]:
# Every code file is one reference list. The first column is named after the list (admission_method_key);
# the other eleven are the same in every file, so rename the first and stack them.
import os

folder = FILES + BRONZE + "nhs_dd_cds/"
names = sorted(f[:-4] for f in os.listdir(folder) if f.endswith(".csv"))

codes = None
for name in names:
    one = (spark.read.option("header", True).csv(f"Files/{BRONZE}nhs_dd_cds/{name}.csv")
                .withColumnRenamed(f"{name}_key", "code_key")
                .withColumn("list_name", F.lit(name))
                .withColumn("is_current", F.col("is_current") == "true"))
    codes = one if codes is None else codes.unionByName(one)

codes = codes.cache()
codes.write.mode("overwrite").saveAsTable(PREFIX + "codes")

lists = (spark.read.option("header", True).csv(f"Files/{BRONZE}lists.csv")
              .withColumn("is_current", F.col("is_current") == "true"))
lists.write.mode("overwrite").saveAsTable(PREFIX + "lists")

print(codes.count(), "code versions from", len(names), "files;", lists.count(), "list versions")

In [ ]:
# One table per reference list, today's codes only — the same tables the Parquet notebook builds.
for name in names:
    (codes.filter((F.col("list_name") == name) & F.col("is_current"))
          .select("code_kind", "code", "description")
          .write.mode("overwrite").saveAsTable(PREFIX + name))

print(len(names), "reference tables written")
display(spark.sql(f"SELECT * FROM {PREFIX}admission_method ORDER BY code_kind DESC, code"))

## Notes

- **Bronze is the untouched file.** `Files/bronze/ukhrd/<date>/` keeps each day's CSVs exactly as published,
  dated, so you can always go back to what arrived.
- **Types.** CSV is all text. This notebook converts `is_current` to a boolean and leaves the rest as text;
  the dates are ISO 8601 (`2026-09-13T13:38:02.114233+00:00`), so `CAST(valid_from AS TIMESTAMP)` works when
  you want real timestamps. The Parquet route gives you those types already.
- **Joining** is the same as the other notebook: `ukhrd_<list>` for today's meaning, `ukhrd_codes` with
  `valid_from` / `valid_to` for the meaning on the date of a record, `ukhrd_lists` for the NHS page.
- **Schedule** it daily. UKHRD checks the dictionary every morning, so the CSVs in the repository are never
  more than a day behind the publisher.